# 02 - Distribution Fitting

**Phase 2 (Weeks 2-3)** of the Portuguese wildfire catastrophe loss model.

Per the revised PRD and what Phase 1 found (see `docs/worklog.md`,
`docs/phase1-writeup.md`):

- **Event unit: fire-day**, not the raw EFFIS polygon. Individual polygons
  on the same day are usually the same weather-driven outbreak, not
  independent ignitions; grouping by start date (`wildfire_model.
  build_fire_day_events`) cuts annual count overdispersion from ~41 to
  ~7 (on the training years) and removes the significant count-vs-fire-size
  correlation found at the polygon level. A residual, smaller dependence
  remains (see below) - flagged, not hidden.
- **Frequency: Poisson vs Negative Binomial.** Fire-day counts are still
  overdispersed even after the fire-day regrouping, so both are fitted
  (by maximum likelihood) and compared on a formal dispersion test and
  on AIC.
- **Severity: Lognormal body, Generalised Pareto tail above a threshold**
  chosen with a mean-excess plot, fitted on `Estimated_Loss_EUR_2025`
  (2025 euros) - equivalent to fitting on burned area and rescaling, since
  loss is area times a constant (see Phase 1's "Loss calibration"), but
  more direct. Not yet implemented (see below).
- **Goodness of fit: Anderson-Darling, QQ plots, AIC/BIC; bootstrap
  p-values**, not Kolmogorov-Smirnov - KS p-values are invalid once
  parameters are fitted on the same data, and the PRD calls this out
  explicitly.
- **Train/hold-out split, confirmed:** fit everything here on **2009-2020**
  only (`wildfire_model.TRAIN_YEARS`). **2021-2025 is reserved for the
  Phase 4 backtest** (`wildfire_model.HOLDOUT_YEARS`) and must not touch
  these fits, or the backtest stops being a real out-of-sample test.
- **Residual dependence not captured by a plain frequency-then-severity
  draw:** even at fire-day granularity, annual burnt area's standard
  deviation is still ~1.8x what independent counts and severities would
  give (see the Phase 1 finding). A year-level severity factor is the
  planned fix for Phase 3, not something this notebook's per-event fits
  capture on their own - noted here so it isn't forgotten by the time
  Phase 3 starts.

**Status: frequency model fitted and decided below (Poisson vs Negative
Binomial). Severity (Lognormal + GPD tail) and goodness-of-fit are the
next step, not yet implemented.**

Success criteria (per the revised PRD): candidate distributions compared
by AIC/BIC, Anderson-Darling with bootstrap p-values, and no systematic
deviation in the top 10% of events on a QQ plot. R^2 and plain KS are not
used.


## Imports

In [1]:
import sys
import json
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.optimize import minimize
import matplotlib.pyplot as plt

from pathlib import Path

sys.path.insert(0, "..")
from wildfire_model import build_fire_day_events, split_train_holdout, TRAIN_YEARS, HOLDOUT_YEARS

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")


## Save fitted parameters

Defined early since several sections below use it.

In [2]:
def save_model_params(params: dict, filename: str) -> None:
    """Persist fitted distribution parameters to models/ as JSON.

    Parameters
    ----------
    params : dict
        Parameter dictionary (e.g. output of fit_poisson_frequency).
    filename : str
        Output filename, written under MODELS_DIR (e.g. "poisson_frequency.json").
    """
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    with open(MODELS_DIR / filename, "w") as f:
        json.dump(params, f, indent=2)


## Load processed data

In [3]:
def load_processed_data(path: Path = PROCESSED_DIR / "wildfires_processed.csv") -> pd.DataFrame:
    """Load the cleaned wildfire dataset produced in 01_eda.ipynb.

    Parameters
    ----------
    path : Path
        Location of the processed CSV.

    Returns
    -------
    pd.DataFrame
        Columns: Date, Location, Burned_Area_ha, Estimated_Loss_EUR,
        Estimated_Loss_EUR_2025, Loss_Source (see 01_eda.ipynb). One row
        per raw EFFIS fire record (30 ha+, mainland, 2009-2025) - not yet
        grouped into fire-day events; see the next section for that.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run notebooks/01_eda.ipynb (or `python main.py --phase 1`) first."
        )
    df = pd.read_csv(path, parse_dates=["Date"])
    return df


## Fire-day events and train/hold-out split

Reproduces the fire-day numbers already quoted in `docs/phase1-writeup.md`
as a check that `wildfire_model.build_fire_day_events` matches what was
described there (it was previously only prose, not committed code - see
`docs/worklog.md`).

In [4]:
processed_df = load_processed_data()
fire_day_events = build_fire_day_events(processed_df)
train_events, holdout_events = split_train_holdout(fire_day_events)

print(f"Fire-day events, {fire_day_events['Date'].dt.year.min()}-{fire_day_events['Date'].dt.year.max()}: {len(fire_day_events)}")
top10_share = fire_day_events.nlargest(10, "Burned_Area_ha")["Burned_Area_ha"].sum() / fire_day_events["Burned_Area_ha"].sum()
print(f"Top 10 fire-days' share of all burnt area: {top10_share:.1%}")
biggest = fire_day_events.loc[fire_day_events["Burned_Area_ha"].idxmax()]
print(f"Largest fire-day: {biggest['Date'].date()}, {int(biggest['N_Fires'])} fires, {biggest['Burned_Area_ha']:,.0f} ha")

print(f"\nTrain {TRAIN_YEARS}: {len(train_events)} fire-day events")
print(f"Holdout {HOLDOUT_YEARS}: {len(holdout_events)} fire-day events (reserved for Phase 4 - not used below)")

train_counts = train_events.groupby(train_events["Date"].dt.year).size().reindex(
    range(TRAIN_YEARS[0], TRAIN_YEARS[1] + 1), fill_value=0
)
print("\nTraining-year fire-day counts:")
print(train_counts.to_string())


Fire-day events, 2009-2025: 1283
Top 10 fire-days' share of all burnt area: 35.1%
Largest fire-day: 2017-10-15, 33 fires, 196,476 ha

Train (2009, 2020): 884 fire-day events
Holdout (2021, 2025): 399 fire-day events (reserved for Phase 4 - not used below)

Training-year fire-day counts:
Date
2009     85
2010     62
2011     84
2012     70
2013     76
2014     25
2015     75
2016     78
2017    120
2018     45
2019     83
2020     81


## Frequency model: Poisson vs Negative Binomial

Fitted on **training-year (2009-2020) fire-day counts only**. The PRD's
technical decision is "Poisson if variance is close to the mean; otherwise
negative binomial", so the dispersion is tested formally before choosing.

In [5]:
def overdispersion_test(counts: np.ndarray) -> dict:
    """Test H0: counts are Poisson-distributed (variance = mean).

    Uses the index-of-dispersion statistic T = sum((x - mean)^2) / mean,
    which is approximately chi-squared distributed with n-1 degrees of
    freedom under H0 (a standard, simple test for overdispersion in count
    data). A large T / small p-value means the data are more variable than
    a Poisson process allows.

    Parameters
    ----------
    counts : np.ndarray
        One value per period (e.g. per year): number of events.

    Returns
    -------
    dict
        {"statistic": float, "df": int, "p_value": float,
         "dispersion_ratio": float} - dispersion_ratio is the sample
        variance-to-mean ratio (1.0 under exact Poisson dispersion).
    """
    counts = np.asarray(counts, dtype=float)
    n, mean = len(counts), counts.mean()
    statistic = float(np.sum((counts - mean) ** 2) / mean)
    df = n - 1
    p_value = float(1 - stats.chi2.cdf(statistic, df=df))
    dispersion_ratio = float(counts.var(ddof=1) / mean)
    return {"statistic": statistic, "df": df, "p_value": p_value, "dispersion_ratio": dispersion_ratio}


In [6]:
def fit_poisson_frequency(annual_counts: np.ndarray) -> dict:
    """Fit a Poisson distribution to annual fire-day counts via MLE.

    Parameters
    ----------
    annual_counts : np.ndarray
        One value per year: number of fire-day events that year.

    Returns
    -------
    dict
        {"lambda": float, "loglik": float, "aic": float} - the MLE rate
        parameter is the sample mean; aic uses 1 fitted parameter.
    """
    annual_counts = np.asarray(annual_counts, dtype=float)
    lam = float(annual_counts.mean())
    loglik = float(stats.poisson.logpmf(annual_counts, lam).sum())
    return {"lambda": lam, "loglik": loglik, "aic": 2 * 1 - 2 * loglik}


In [7]:
def fit_negative_binomial_frequency(annual_counts: np.ndarray) -> dict:
    """Fit a Negative Binomial distribution to annual fire-day counts via MLE.

    Parameterised as scipy.stats.nbinom(r, p) (r = number of "successes",
    p = success probability; mean = r(1-p)/p, var = r(1-p)/p^2). The
    method-of-moments solution (matching the sample mean and variance
    exactly) is used as the optimiser's starting point, since it is a
    good, stable estimate for a two-parameter fit on a short (n=12) annual
    series, then refined by MLE.

    Parameters
    ----------
    annual_counts : np.ndarray
        One value per year: number of fire-day events that year.

    Returns
    -------
    dict
        {"r": float, "p": float, "mean": float, "loglik": float,
         "aic": float} - aic uses 2 fitted parameters.
    """
    annual_counts = np.asarray(annual_counts, dtype=float)
    mean, var = annual_counts.mean(), annual_counts.var(ddof=1)
    if var <= mean:
        raise ValueError("Sample variance <= mean; data are not overdispersed, use Poisson instead.")
    r0, p0 = mean ** 2 / (var - mean), mean / var

    def negloglik(params):
        r, p = params
        if r <= 0 or not (0 < p < 1):
            return np.inf
        return -stats.nbinom.logpmf(annual_counts, r, p).sum()

    result = minimize(negloglik, x0=[r0, p0], method="Nelder-Mead")
    r, p = result.x
    loglik = float(-result.fun)
    return {"r": float(r), "p": float(p), "mean": float(r * (1 - p) / p), "loglik": loglik, "aic": 2 * 2 - 2 * loglik}


### Run: frequency model

In [8]:
dispersion = overdispersion_test(train_counts.values)
print(f"Dispersion test (H0: Poisson): statistic={dispersion['statistic']:.1f}, df={dispersion['df']}, "
      f"p={dispersion['p_value']:.2e}, variance/mean={dispersion['dispersion_ratio']:.2f}")

poisson_params = fit_poisson_frequency(train_counts.values)
negbin_params = fit_negative_binomial_frequency(train_counts.values)

print(f"\nPoisson:            lambda={poisson_params['lambda']:.2f}  "
      f"loglik={poisson_params['loglik']:.2f}  AIC={poisson_params['aic']:.2f}")
print(f"Negative Binomial:  r={negbin_params['r']:.2f}  p={negbin_params['p']:.3f}  "
      f"mean={negbin_params['mean']:.2f}  loglik={negbin_params['loglik']:.2f}  AIC={negbin_params['aic']:.2f}")

frequency_model = "negative_binomial" if (dispersion["p_value"] < 0.05 and negbin_params["aic"] < poisson_params["aic"]) else "poisson"
print(f"\nDispersion test p={dispersion['p_value']:.1e} (<<0.05: fire-day counts are not Poisson-distributed),")
print(f"and Negative Binomial's AIC is {poisson_params['aic'] - negbin_params['aic']:.0f} points lower.")
print(f"Decision: frequency_model = \"{frequency_model}\"")

save_model_params(poisson_params, "poisson_frequency.json")
save_model_params(negbin_params, "negative_binomial_frequency.json")
save_model_params({"chosen": frequency_model, "dispersion_test": dispersion}, "frequency_model_choice.json")


Dispersion test (H0: Poisson): statistic=79.9, df=11, p=1.52e-12, variance/mean=7.27

Poisson:            lambda=73.67  loglik=-80.53  AIC=163.05
Negative Binomial:  r=10.28  p=0.122  mean=73.67  loglik=-55.13  AIC=114.26

Dispersion test p=1.5e-12 (<<0.05: fire-day counts are not Poisson-distributed),
and Negative Binomial's AIC is 49 points lower.
Decision: frequency_model = "negative_binomial"


## Severity model: Lognormal + Pareto tail

**Not yet implemented.** Next step: fit a Lognormal body to
`Estimated_Loss_EUR_2025` on `train_events` (fire-day totals, training
years only), choose a Generalised Pareto tail threshold via a mean-excess
plot, and fit the tail to exceedances above it.

In [9]:
def fit_lognormal_severity(losses: np.ndarray) -> dict:
    """Fit a Lognormal distribution to per-event loss magnitudes via MLE.

    Parameters
    ----------
    losses : np.ndarray
        Per-event estimated loss amounts (EUR, 2025 prices - see
        Estimated_Loss_EUR_2025), training years only.

    Returns
    -------
    dict
        {"shape": float, "loc": float, "scale": float} from
        scipy.stats.lognorm.fit(losses, floc=0).
    """
    # TODO: implement with scipy.stats.lognorm.fit(losses, floc=0)
    raise NotImplementedError("Lognormal fitting is the next step - see notebook intro")


In [10]:
def fit_pareto_tail(losses: np.ndarray, threshold: float) -> dict:
    """Fit a Generalized Pareto distribution to losses exceeding a threshold.

    Parameters
    ----------
    losses : np.ndarray
        Per-event estimated loss amounts (EUR, 2025 prices), training
        years only.
    threshold : float
        Loss level above which the tail fit is applied (peaks-over-
        threshold); choose with a mean-excess plot, not an arbitrary
        percentile.

    Returns
    -------
    dict
        {"shape": float, "loc": float, "scale": float, "threshold": float}
        from scipy.stats.genpareto.fit on exceedances.
    """
    # TODO: implement with scipy.stats.genpareto.fit on (losses[losses > threshold] - threshold)
    raise NotImplementedError("Pareto tail fitting is the next step - see notebook intro")


## Goodness-of-fit tests

**Not yet implemented.** Per the revised PRD: Anderson-Darling (not KS -
KS p-values are invalid once parameters are fitted on the same data),
with bootstrap p-values since scipy's built-in critical values assume
known parameters, not fitted ones. QQ plots and AIC/BIC comparisons
belong here too.

In [11]:
def run_anderson_darling_test(data: np.ndarray, dist: str = "norm") -> dict:
    """Run an Anderson-Darling goodness-of-fit test.

    Parameters
    ----------
    data : np.ndarray
        Observed sample. For Lognormal fits, pass log(losses) with dist="norm"
        (scipy.stats.anderson does not support lognorm or genpareto directly).
    dist : str
        Distribution family supported by scipy.stats.anderson (e.g. "norm", "expon").

    Returns
    -------
    dict
        {"statistic": float, "critical_values": list, "significance_levels": list}
    """
    # TODO: implement with scipy.stats.anderson(data, dist=dist)
    raise NotImplementedError("Anderson-Darling test is the next step - see notebook intro")


In [12]:
def bootstrap_ad_pvalue(data: np.ndarray, dist, fit_func, n_boot: int = 1000, seed: int = 0) -> dict:
    """Bootstrap an Anderson-Darling p-value for a distribution whose parameters
    were fitted on this same data (e.g. the Generalised Pareto tail, which
    scipy.stats.anderson does not support directly).

    Repeatedly simulates a same-size sample from the fitted distribution,
    refits it, and computes the AD statistic each time, to build the null
    distribution of the AD statistic under "parameters estimated from the
    data" rather than "parameters known in advance" (the assumption behind
    scipy's built-in critical values).

    Parameters
    ----------
    data : np.ndarray
        Observed sample the distribution was fitted to.
    dist : scipy.stats rv_continuous
        Distribution object (e.g. scipy.stats.genpareto).
    fit_func : callable
        Function that takes an array and returns fitted params as a tuple
        compatible with dist.cdf(x, *params).
    n_boot : int
        Number of bootstrap replicates.
    seed : int
        Random seed, for reproducibility (PRD requirement).

    Returns
    -------
    dict
        {"statistic": float, "p_value": float, "n_boot": int}
    """
    # TODO: implement the parametric bootstrap described above
    raise NotImplementedError("Bootstrap AD p-value is the next step - see notebook intro")


## Diagnostic plots

In [13]:
def plot_empirical_vs_fitted_cdf(data: np.ndarray, dist, params: tuple, ax=None, label: str = "Fitted"):
    """Overlay the empirical CDF against a fitted distribution's CDF.

    Parameters
    ----------
    data : np.ndarray
        Observed sample.
    dist : scipy.stats rv_continuous
        Distribution object (e.g. scipy.stats.lognorm).
    params : tuple
        Fitted parameters to pass to dist.cdf.
    ax : matplotlib.axes.Axes, optional
        Axes to plot on; a new figure/axes is created if omitted.
    label : str
        Legend label for the fitted curve.
    """
    if ax is None:
        _, ax = plt.subplots()
    sorted_data = np.sort(data)
    empirical_cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
    ax.plot(sorted_data, empirical_cdf, label="Empirical")
    ax.plot(sorted_data, dist.cdf(sorted_data, *params), label=label, linestyle="--")
    ax.set_xlabel("Value")
    ax.set_ylabel("Cumulative probability")
    ax.legend()
    return ax


In [14]:
def plot_mean_excess(losses: np.ndarray, ax=None):
    """Plot a mean-excess plot, used to choose the Generalised Pareto tail threshold.

    For a range of candidate thresholds u, plots the mean of (loss - u)
    over losses exceeding u. A roughly linear region indicates a range of
    u for which a GPD tail is a reasonable model; the threshold is chosen
    from where that linearity starts.

    Parameters
    ----------
    losses : np.ndarray
        Per-event estimated loss amounts.
    ax : matplotlib.axes.Axes, optional
        Axes to plot on; a new figure/axes is created if omitted.
    """
    if ax is None:
        _, ax = plt.subplots()
    thresholds = np.sort(losses)[:-5]  # leave enough points above the highest threshold to average
    mean_excess = [losses[losses > u].mean() - u for u in thresholds]
    ax.plot(thresholds, mean_excess)
    ax.set_xlabel("Threshold (EUR)")
    ax.set_ylabel("Mean excess over threshold")
    return ax


## Run distribution fitting

Frequency is fitted above. Severity, goodness-of-fit and the diagnostic
plots are pending (see "Status" in the notebook intro) - this cell will
run and save them once implemented.

In [15]:
# TODO: once fit_lognormal_severity / fit_pareto_tail / run_anderson_darling_test /
# bootstrap_ad_pvalue are implemented, run them here on train_events["Estimated_Loss_EUR_2025"]
# and save_model_params(...) the results, alongside the frequency params already saved above.

# losses = train_events["Estimated_Loss_EUR_2025"].values
# plot_mean_excess(losses); plt.show()
# lognormal_params = fit_lognormal_severity(losses)
# pareto_params = fit_pareto_tail(losses, threshold=...)  # from the mean-excess plot, not a fixed percentile
# ad_result = run_anderson_darling_test(np.log(losses))
# plot_empirical_vs_fitted_cdf(losses, stats.lognorm, tuple(lognormal_params.values()))
# save_model_params(lognormal_params, "lognormal_severity.json")
# save_model_params(pareto_params, "pareto_tail.json")


## Parameter estimates summary

**Frequency (fitted above, training years 2009-2020):**
- Dispersion test: fire-day counts reject Poisson decisively
  (p << 0.05); variance/mean ~7 on the training years.
- **Chosen model: Negative Binomial** (see `models/frequency_model_choice.json`
  once run) - lower AIC than Poisson by a wide margin, consistent with the
  dispersion test.
- Parameters saved to `models/poisson_frequency.json` and
  `models/negative_binomial_frequency.json` (both kept, for comparison).

**Severity (Lognormal body, Generalised Pareto tail):** not yet fitted -
see "Run distribution fitting" above.

**Goodness-of-fit:** not yet run.
